In [65]:
import yaml
import PyPDF2
import re
import csv
import os


In [66]:
# Loading data
with open ('config.yaml', 'r') as file_adress:
    config = yaml.safe_load(file_adress)

pdfFile = config['resultpdf']

In [67]:
pattern1 = r'^[a-zA-Z]+' 

pattern2 = r'\s+[a-zA-Z]+-[a-zA-Z]+\s?[[a-zA-Z]+]?\s+[a-zA-Z]{1,2}\s{1}[a-zA-Z]+/[a-zA-Z]+\s+<?>?[\s{1}]?\d+,?[\d+]?\s+\d+,?[\d+]?\s-\s+\d+,?[\d+]?' 
  
pattern3 = r'\s+[a-zA-Z]+/[a-zA-Z]+-[a-zA-Z]+\s+\d+,?[[\d]+]?\s+\d+,?[[\d]+]?\s-\s+\d+,?[[\d]+]?'

pattern4 = r'\s+[a-zA-Z]+\s\([a-zA-Z]+\)\s+\d+,?[\d+]?\s+<?>?[\s{1}]?\d+,?[\d+]?\s?-?\s?[\d+]?,?[\d+]?'

pattern5 = r'\s+[a-zA-Z]{1,2}-?[a-zA-Z]+\s?\(?.+?\)?\s+%\s+\d+,?\d*\s*<?>?\s?\d*,?\d*\s?-?\s?\d*,?\d*'

#pattern5 = r'\s+[a-zA-Z]{1,2}-?[a-zA-Z]+\s?\(?.+?\)?\s+%\s+\d+,?[\d+]?'
pattern6 = r'\s+[a-zA-Z]{1,2}-?[a-zA-Z]+[[\s]+]?[a-zA-Z]*[[\s]+]?%\s+<?\s?\d+,?\s?[[\d]+]?\s+<?>?\s?\d+,?[[\d]+]?\s?-?\s?\d*,?\d*'

pattern7 = r'\s+\w+\s?[a-zA-Z]*\s+[a-zA-Z]+\s?[a-zA-Z]*/?[a-zA-Z]*\s+\d+,?\d*\s+\d+,?\d*\s?-?\s?\d+,?\d*'

pattern8 = r'\s+[a-zA-Z]+-?[a-zA-Z]*\s\([a-zA-Z]+\)\s+[a-zA-Z]+\+/[a-zA-Z]+\s+\d+,?\d*\s+>?<?\s?\d+,?\d*'

#pattern? = r'\s+[a-zA-Z]{1,2}/?[a-zA-Z]{0,2}-?[[a-zA-Z]+]?\s?\(?.+?\)?\s+\d+,?[\d+]?\s+\d+,?[\d+]?\s?-?\s?[\d+]?,?[\d+]?'
 
 
# Open the PDF file
pdf_file = open(pdfFile, 'rb')

# Create a PDF reader object
pdf_reader = PyPDF2.PdfFileReader(pdf_file)

def str_cleaner(str):
    result_line = str.strip().split('  ') 
    while '' in result_line:
        result_line.remove( '')
    for index in range (len(result_line)):
        result_line[index] = result_line[index].strip()
    return result_line    


def cleaner(list):

    line_list = []
    for match in list:
        result_line = str_cleaner(match)
        line_list.append(result_line)     
        
    return line_list
 
def organizer(line):

    line.insert(0,'')
    return line 

# Search for the "Resultaat" keyword and extract the text following it
with open('output.csv', 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    for page in pdf_reader.pages:
        page_text = page.extract_text()
        if 'Resultaat' in page_text:
            lines = page_text.split('\n')
            for i, line in enumerate(lines):
                if 'Resultaat' in line: 
                    header = line 
                    result_start = i + 1
                    break
            row = str_cleaner(header) 
            if len(row)<6: 
                header = lines[result_start] 
                row = str_cleaner(header)
            if row[0]!='Resultaat':
                row.insert(0,'Resultaat')
            row.insert(1,'Mineral')    
            writer.writerow(row)   
             
            for line in lines[result_start:]: 
                matches = re.findall(pattern1, line)
                matches2 = re.findall(pattern2, line)
                matches3 = re.findall(pattern3, line)
                matches4 = re.findall(pattern4, line)
                matches5 = re.findall(pattern5, line)
                matches6 = re.findall(pattern6,line)  
                matches7 = re.findall(pattern7,line)
                matches8 = re.findall(pattern8,line) 

                if matches:
                    writer.writerow(matches)

                if matches2:
                    result_line = cleaner(matches2)
                    for line in result_line:
                        row = organizer(line)  
                        writer.writerow(row) 


                elif matches3:
                    result_line = cleaner(matches3)
                    for line in result_line:
                        row = organizer(line)
                        row.insert(2,'')      
                        writer.writerow(row)
                        
                elif matches4:
                    result_line = cleaner(matches4)
                    for line in result_line:
                        row = organizer(line)
                        row.insert(2,'')      
                        writer.writerow(row)


                elif matches5:
                    result_line = cleaner(matches5)
                    for line in result_line:
                        row = organizer(line)
                        if 'Organische stof' in row:
                            if len(row)>4:
                                row = row[:4]   
                        writer.writerow(row)

                elif matches6:
                    result_line = cleaner(matches6)
                    for line in result_line:
                        row = organizer(line)   
                        writer.writerow(row)

                elif matches7:
                    result_line = cleaner(matches7)
                    for line in result_line:
                        row = organizer(line)   
                        writer.writerow(row)

                elif matches8:
                    result_line = cleaner(matches8)
                    for line in result_line:
                        row = organizer(line)   
                        writer.writerow(row)       
            break              
            

Xref table not zero-indexed. ID numbers for objects will be corrected.


In [68]:
print(lines[result_start-1:result_start+3])

['Resultaat                                                                                                                                 Reparatie', '                                            Eenheid        Resu ltaat      Streeftraject        laag  vrij laag   goed  vrij hoo g  hoog     advies (kg/ha)', 'Chemisch', '                 N-totale bodemvoorraad     kg N/ha        5790            3400 - 4970        ']


In [69]:
 
# Open the input CSV file
with open('output.csv', 'r') as input_file:
    # Open the output CSV file for writing
    with open('output_file.csv', 'w', newline='') as output_file:
        # Create CSV reader and writer objects
        csv_reader = csv.reader(input_file)
        csv_writer = csv.writer(output_file)

        header = next(csv_reader)
        csv_writer.writerow(header)

            
        previous_title = ''    
        # Iterate over each row in the input CSV file
        for row in csv_reader: 

            # Assuming the second column needs to be validated
            column1 = row[0]  

            # Check if column2 meets your validation criteria
            if column1 != '':
                previous_title = column1
            else:
                row[0] = previous_title    

            # Write the modified row to the output CSV file
            csv_writer.writerow(row)


In [70]:


# Specify the path to the CSV file
file_path = 'output.csv'
os.remove(file_path) 